# 17.4 零點偏移有什麼作用？

# 第 17 章：量化，用更少位元表示模型

前置：第2章Linear與數值。像把連續刻度尺改成較粗的格子：先看誤差，再看真正的儲存bytes，最後才談硬體加速。quantization.py的參考Linear會反量化，不會假裝是低位元kernel。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：換較粗刻度尺：省格子但有誤差**

縮小儲存與硬體加速分開驗證。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/quantization.svg")))

**先想一想：**輸入都在2到3，爲什麼只用對稱範圍可能浪費格子？

範圍不以0爲中心時，zero-point把整數0格搬到合適位置。先讓浮點0能還原，並保存scale與zero-point。

**把直覺寫成數學：**q=round(x/s)+z；x_hat=(q−z)s；affine min/max需包括0。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.quantization import quantize_affine

x = torch.tensor([0.0, 0.2, 1.0, 3.0])
q, s, z = quantize_affine(x, bits=4)
restored = (q.float() - z) * s
print(q, s, z, restored)
assert restored[0].abs() < 1e-6

**如何讀結果：**這份參考用unsigned 0..15；int4 packing章節用signed值，兩個表示不要混。

**只改一件事：**只把範圍改成−3到1，觀察zero-point移動。
